In [ ]:
import sys, torch

print(f"python  {sys.version.split()[0]}")
print(f"torch   {torch.__version__}")
print(f"cuda    {torch.version.cuda}")
print(f"gpu ok  {torch.cuda.is_available()}")

In [ ]:
a = torch.randn(4000, 4000, device="cuda")
print(f"matmul  ok   {(a @ a).sum().item():.1f}")

with torch.autocast("cuda", dtype=torch.float16):
    v = (a @ a).mean().item()
print(f"amp     ok   {v:.4f}")

del a; torch.cuda.empty_cache()

In [ ]:
import ultralytics
print(f"ultralytics {ultralytics.__version__}")
print("\nReady. Move on to notebook 01.")

# 01 Ã¢â‚¬â€ Prepare the data

Turn the competition CSV into YOLO format, and build a split that will tell
us the truth about generalisation.

**Get the data first** (needs a Kaggle account, and you must click
"Join Competition" once to accept the rules):

```
pip install kaggle
kaggle competitions download -c global-wheat-detection -p data/raw
cd data/raw && unzip global-wheat-detection.zip
```

Browser alternative: https://www.kaggle.com/competitions/global-wheat-detection/data

In [ ]:
import ast, shutil
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image


def find_repo_root(start):
    for candidate in (start, *start.parents):
        if (candidate / "data").is_dir() and (candidate / "notebooks").is_dir():
            return candidate
    raise FileNotFoundError("Run this notebook from inside the project repository.")


REPO_ROOT = find_repo_root(Path.cwd().resolve())
RAW = REPO_ROOT / "data" / "raw"
OUT = REPO_ROOT / "data" / "yolo"
IMG = RAW / "train"

# Held out entirely from training. The model never sees these farms, so
# scores on them estimate how it behaves somewhere it has never been.
OOD_SOURCES = ["usask_1", "ethz_1"]

MAX_BOX_AREA = 200_000  # bigger than this is an annotation error
MIN_BOX_SIDE = 2        # pixels
VAL_FRAC = 0.10
SEED = 42

COPY_IMAGES = True  # False = symlink (Linux). True = copy (safe on Windows)

## Load

In [ ]:
df = pd.read_csv(RAW / "train.csv")
print(f"{len(df):,} annotation rows across {df.image_id.nunique():,} images")
df.head()

## The bbox column is a string

Look at the type below. `bbox` is **text**, not a list Ã¢â‚¬â€ and the four numbers
are `[x_min, y_min, width, height]`, i.e. a corner plus a size.

YOLO wants something different: `[x_center, y_center, width, height]`, all
**divided by the image size** so everything sits between 0 and 1.

This conversion is where most people introduce a silent bug. Nothing errors Ã¢â‚¬â€
the numbers just end up in the wrong slots and the model learns garbage.

In [ ]:
print(type(df.bbox.iloc[0]))
print(df.bbox.iloc[0])

In [ ]:
# Vectorised split - far faster than ast.literal_eval on 148k rows
parts = df["bbox"].str.strip("[]").str.split(",", expand=True).astype(float)
parts.columns = ["x", "y", "w", "h"]
df = pd.concat([df.drop(columns=["bbox"]), parts], axis=1)
df.head(3)

## Clean the boxes

Real datasets are messy and this one is no exception. Some boxes run past the
image edge, a handful are absurdly large (annotation slips), and a few have
near-zero width. Clip first, then drop what is left over.

In [ ]:
n0 = len(df)

# corner form makes clipping easy
df["x1"] = df["x"].clip(lower=0)
df["y1"] = df["y"].clip(lower=0)
df["x2"] = np.minimum(df["x"] + df["w"], df["width"])
df["y2"] = np.minimum(df["y"] + df["h"], df["height"])

df["w"] = df["x2"] - df["x1"]
df["h"] = df["y2"] - df["y1"]
df["area"] = df["w"] * df["h"]

too_small = (df["w"] < MIN_BOX_SIDE) | (df["h"] < MIN_BOX_SIDE)
too_big   = df["area"] > MAX_BOX_AREA

print(f"dropping {too_small.sum()} degenerate, {too_big.sum()} oversized")
df = df[~(too_small | too_big)].copy()
print(f"{n0:,} -> {len(df):,} boxes")

## What is actually in here

In [ ]:
per_src = df.groupby("source").agg(images=("image_id", "nunique"),
                                   boxes=("image_id", "size"))
per_src["boxes_per_image"] = (per_src.boxes / per_src.images).round(1)
per_src["median_box_px"]   = df.groupby("source")["area"].median().pow(0.5).round(0)
per_src

In [ ]:
counts = df.groupby("image_id").size()
side   = np.sqrt(df["area"])

fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
ax[0].hist(counts, bins=50, color="#4a7c59")
ax[0].set_title("wheat heads per image"); ax[0].set_xlabel("count")
ax[1].hist(side, bins=60, range=(0, 250), color="#8b6f47")
ax[1].set_title("box size (sqrt of area, px)"); ax[1].set_xlabel("pixels")
plt.tight_layout(); plt.show()

print(f"heads per image  mean {counts.mean():.0f}  median {counts.median():.0f}  max {counts.max()}")
print(f"box side px      median {side.median():.0f}  p5 {side.quantile(.05):.0f}  p95 {side.quantile(.95):.0f}")

## Look at the domain shift

This is the cell that makes the whole project make sense. One image from each
source, side by side.

Notice how different they are Ã¢â‚¬â€ lighting, soil colour, crop maturity, camera
distance, head density. Now remember the competition's hidden test set came
from **Australia, Japan and China**, none of which appear below.

That mismatch is the actual problem. Not detection.

In [ ]:
sources = sorted(df.source.unique())
fig, axes = plt.subplots(1, len(sources), figsize=(3 * len(sources), 3.4))

for ax, src in zip(np.atleast_1d(axes), sources):
    iid = df[df.source == src].image_id.iloc[0]
    ax.imshow(Image.open(IMG / f"{iid}.jpg"))
    ax.set_title(src, fontsize=10)
    ax.axis("off")

plt.tight_layout(); plt.show()

## Sanity check the annotations by drawing them

In [ ]:
import matplotlib.patches as patches

def show_boxes(image_id, ax=None):
    g = df[df.image_id == image_id]
    if ax is None:
        _, ax = plt.subplots(figsize=(7, 7))
    ax.imshow(Image.open(IMG / f"{image_id}.jpg"))
    for r in g.itertuples():
        ax.add_patch(patches.Rectangle((r.x1, r.y1), r.w, r.h,
                                       fill=False, edgecolor="#00ff41", lw=1.2))
    ax.set_title(f"{image_id}  |  {g.source.iloc[0]}  |  {len(g)} heads", fontsize=10)
    ax.axis("off")

show_boxes(df.image_id.iloc[0])
plt.show()

## Build the splits

**This is the most important design decision in the project.**

A random 80/20 split would tell you the model is ~97% accurate and it would be
lying, because the real test images came from different continents. So instead:

| split | what it is | purpose |
|---|---|---|
| `train` | 5 sources | learning |
| `val` | random 10% of those same 5 | monitoring, early stopping |
| `ood` | `usask_1` + `ethz_1`, held out whole | the honest number |

The gap between `val` and `ood` is the number this entire project exists to
shrink. Everything you try in week 2 gets measured against it.

In [ ]:
# YOLO format: class x_center y_center width height, all normalised 0-1
df["xc"] = ((df.x1 + df.x2) / 2) / df.width
df["yc"] = ((df.y1 + df.y2) / 2) / df.height
df["nw"] = df.w / df.width
df["nh"] = df.h / df.height

all_imgs = sorted(p.stem for p in IMG.glob("*.jpg"))
src_of   = df.drop_duplicates("image_id").set_index("image_id")["source"].to_dict()

ood_ids  = [i for i in all_imgs if src_of.get(i) in OOD_SOURCES]
pool_ids = [i for i in all_imgs if src_of.get(i) not in OOD_SOURCES]

rng = np.random.default_rng(SEED)
pool_ids = list(rng.permutation(pool_ids))
n_val    = int(len(pool_ids) * VAL_FRAC)
val_ids, train_ids = pool_ids[:n_val], pool_ids[n_val:]

print(f"train {len(train_ids):5d}   val {len(val_ids):5d}   ood {len(ood_ids):5d}")

In [ ]:
def write_split(image_ids, split):
    img_dir = OUT / "images" / split
    lbl_dir = OUT / "labels" / split
    img_dir.mkdir(parents=True, exist_ok=True)
    lbl_dir.mkdir(parents=True, exist_ok=True)

    grouped, n_box, n_empty = dict(tuple(df.groupby("image_id"))), 0, 0

    for iid in image_ids:
        src = IMG / f"{iid}.jpg"
        if not src.exists():
            continue
        dst = img_dir / f"{iid}.jpg"
        if COPY_IMAGES:
            if not dst.exists():
                shutil.copy2(src, dst)
        else:
            if dst.exists() or dst.is_symlink():
                dst.unlink()
            dst.symlink_to(src.resolve())

        lines = []
        if iid in grouped:
            for r in grouped[iid].itertuples():
                lines.append(f"0 {r.xc:.6f} {r.yc:.6f} {r.nw:.6f} {r.nh:.6f}")
            n_box += len(lines)
        else:
            n_empty += 1   # background image - an empty label file is correct
        (lbl_dir / f"{iid}.txt").write_text("\n".join(lines))

    print(f"{split:6s} {len(image_ids):5d} images  {n_box:7,} boxes  {n_empty} background-only")

for ids, name in [(train_ids, "train"), (val_ids, "val"), (ood_ids, "ood")]:
    write_split(ids, name)

In [ ]:
root = OUT.resolve()
(OUT / "wheat.yaml").write_text(
    f"path: {root}\ntrain: images/train\nval: images/val\nnc: 1\nnames: ['wheat_head']\n")
(OUT / "wheat_ood.yaml").write_text(
    f"path: {root}\ntrain: images/train\nval: images/ood\nnc: 1\nnames: ['wheat_head']\n")

print((OUT / "wheat.yaml").read_text())

## Verify before training

Cheap insurance. If the conversion had a bug, this catches it now instead of
after a wasted overnight run.

In [ ]:
bad = tot = 0
for f in (OUT / "labels").rglob("*.txt"):
    for line in f.read_text().splitlines():
        p = line.split(); tot += 1
        cls = int(p[0]); xc, yc, w, h = map(float, p[1:])
        if cls != 0 or not all(0 <= v <= 1 for v in (xc, yc, w, h)):
            bad += 1
        if xc - w/2 < -1e-6 or xc + w/2 > 1 + 1e-6 or yc - h/2 < -1e-6 or yc + h/2 > 1 + 1e-6:
            bad += 1

print(f"{tot:,} labels checked, {bad} invalid")
assert bad == 0, "conversion bug - do not train until this is 0"

In [ ]:
# Round-trip one box back to pixels and confirm it matches the original CSV
raw = pd.read_csv(RAW / "train.csv")
r   = raw.iloc[0]
x, y, w, h = ast.literal_eval(r.bbox)

lbl = next(p for p in (OUT / "labels").rglob(f"{r.image_id}.txt"))
xc, yc, nw, nh = map(float, lbl.read_text().splitlines()[0].split()[1:])

print(f"original   x={x:7.1f}  y={y:7.1f}  w={w:6.1f}  h={h:6.1f}")
print(f"recovered  x={(xc-nw/2)*1024:7.1f}  y={(yc-nh/2)*1024:7.1f}  "
      f"w={nw*1024:6.1f}  h={nh*1024:6.1f}")

Data is ready. Go to Training

# 02 - Train YOLO11s

The baseline starts from `weights/yolo11s.pt`.

**8 GB VRAM notes.** Measured on this machine: `batch=4` at `imgsz=1024` peaks at **3.1 GB**. If you hit out-of-memory, reduce batch size before reducing image size: wheat heads are small and resolution is important.

**Where runs are saved.** The notebook writes directly to:

```
<repository>/runs/detect/<name>
```

Read `model.trainer.save_dir` after training instead of constructing a checkpoint path manually.

**Notebook caveat.** A long run inside Jupyter is fragile. Checkpoints are written every epoch to `<save_dir>/weights/last.pt`, so a run can resume with `resume=True`.

In [ ]:
from pathlib import Path
import torch
from ultralytics import YOLO

DATA = str(OUT / "wheat.yaml")
DATA_OOD = str(OUT / "wheat_ood.yaml")
MODEL_WEIGHTS = REPO_ROOT / "weights" / "yolo11s.pt"
RUNS_DIR = REPO_ROOT / "runs" / "detect"

IMGSZ = 1024
BATCH = 4
WORKERS = 2  # set 0 if the dataloader hangs (happens on Windows + Jupyter)

if not MODEL_WEIGHTS.exists():
    raise FileNotFoundError(f"Missing pretrained weights: {MODEL_WEIGHTS}")

print(torch.cuda.get_device_name(0))
print(f"{torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## Smoke test first

Three epochs, roughly ten minutes. Its only job is to prove the memory fits and
nothing crashes, before you commit to an overnight run.

In [ ]:
model = YOLO(MODEL_WEIGHTS)

model.train(
    data=DATA, epochs=3, imgsz=IMGSZ, batch=BATCH, workers=WORKERS,
    device=0, amp=True, cache=False,
    project=str(RUNS_DIR), name="smoke", exist_ok=True, seed=42,
)

print(f"\npeak VRAM {torch.cuda.max_memory_allocated() / 1e9:.2f} GB")

## The real run

### Why these augmentation settings

**`mosaic=1.0`** Ã¢â‚¬â€ stitches four training images into one. This was the single
biggest win for the teams that placed well, because it constantly shows the
model wheat at mismatched scales against unfamiliar backgrounds. Cheap domain
robustness.

**`close_mosaic=10`** Ã¢â‚¬â€ turns mosaic off for the final 10 epochs so the model
settles on realistic, unstitched images before finishing.

**`hsv_s`, `hsv_v` high** Ã¢â‚¬â€ colour jitter matters more here than usual. The
unseen domains differ in lighting, soil tone and crop maturity, so teaching the
model to ignore colour is direct preparation for that shift.

**`flipud=0.5`, `degrees=15`** Ã¢â‚¬â€ these are overhead field shots. There is no
"up", so vertical flips and rotation are physically sensible.

In [ ]:
model = YOLO(MODEL_WEIGHTS)

results = model.train(
    data=DATA, epochs=60, imgsz=IMGSZ, batch=BATCH, workers=WORKERS,
    device=0, amp=True, cache=False, patience=20,
    mosaic=1.0, close_mosaic=10, mixup=0.1,
    hsv_h=0.015, hsv_s=0.7, hsv_v=0.4,
    fliplr=0.5, flipud=0.5, degrees=15, scale=0.5, translate=0.1,
    project=str(RUNS_DIR), name="baseline", exist_ok=True, plots=True, seed=42,
)

SAVE_DIR = Path(model.trainer.save_dir)
BEST = SAVE_DIR / "weights" / "best.pt"
print(f"\nsaved to  {SAVE_DIR}")
print(f"best.pt   {BEST}  exists={BEST.exists()}")

## Evaluate twice

Once on familiar farms, once on farms the model has never seen.

In [ ]:
# Falls back to the baseline checkpoint if the kernel restarted after training.
try:
    BEST
except NameError:
    BEST = RUNS_DIR / "baseline" / "weights" / "best.pt"

assert BEST.exists(), f"no weights at {BEST} - has the real run finished?"
best = YOLO(BEST)

ind = best.val(data=DATA, imgsz=IMGSZ, batch=BATCH, split="val",
               project=str(RUNS_DIR), name="baseline_ind", exist_ok=True)
ood = best.val(data=DATA_OOD, imgsz=IMGSZ, batch=BATCH, split="val",
               project=str(RUNS_DIR), name="baseline_ood", exist_ok=True)

print(f"\n{'':10} {'mAP50':>8} {'mAP50-95':>10}")
print(f"{'val  ':10} {ind.box.map50:8.3f} {ind.box.map:10.3f}   familiar farms")
print(f"{'ood  ':10} {ood.box.map50:8.3f} {ood.box.map:10.3f}   unseen farms")
print(f"{'gap  ':10} {ind.box.map50 - ood.box.map50:8.3f} {ind.box.map - ood.box.map:10.3f}   <- what week 2 shrinks")

In [ ]:
import pandas as pd

tbl = pd.DataFrame({
    "mAP50":    [ind.box.map50, ood.box.map50],
    "mAP50-95": [ind.box.map,   ood.box.map],
    "precision":[ind.box.mp,    ood.box.mp],
    "recall":   [ind.box.mr,    ood.box.mr],
}, index=["in-domain", "out-of-domain"]).round(4)

tbl.loc["gap"] = (tbl.loc["in-domain"] - tbl.loc["out-of-domain"]).round(4)
tbl

**That `gap` row is your baseline.** Write it down. Every experiment from here Ã¢â‚¬â€
synthetic data, pseudo-labelling, test-time augmentation Ã¢â‚¬â€ gets judged on
whether it shrinks that number, not on whether it raises in-domain accuracy.

A model that improves in-domain and leaves the gap untouched has learned
nothing useful about generalisation.

## Look at the failures

In [ ]:
import matplotlib.pyplot as plt
from pathlib import Path

# OUT is absolute; "data/yolo/..." would resolve against this notebook's folder.
ood_imgs = sorted((OUT / "images" / "ood").glob("*.jpg"))[:4]
assert ood_imgs, f"no images under {OUT / 'images' / 'ood'}"

preds = best.predict(ood_imgs, imgsz=IMGSZ, conf=0.3, verbose=False)

fig, axes = plt.subplots(1, len(preds), figsize=(4.5 * len(preds), 4.5))
for ax, p in zip(axes, preds):
    ax.imshow(p.plot()[:, :, ::-1])       # BGR -> RGB
    ax.set_title(f"{Path(p.path).stem}  |  {len(p.boxes)} detected", fontsize=9)
    ax.axis("off")
plt.tight_layout(); plt.show()

Compare these against the ground-truth counts you saw in notebook 01. Missed
heads, doubled boxes and false positives on soil or leaves are all worth
noticing Ã¢â‚¬â€ they tell you what the synthetic data in week 2 needs to fix.

## Record before moving on

| | mAP50 | mAP50-95 |
|---|---|---|
| in-domain | | |
| out-of-domain | | |
| **gap** | | |

Peak VRAM: ___ GB Ã‚Â· epoch time: ___ min Ã‚Â· total: ___ h